# 02 — Ingestão Bronze: Previsão 5 Dias (Forecast)
## Módulo 1B | Camada Bronze | Seção do Exame: S2 (21%)

**O que este notebook faz:**
- Chama o endpoint `/forecast` da OpenWeatherMap para 4 cidades
- Retorna JSON com array `list[]` de até 40 previsões (intervalos de 3h × 5 dias)
- Grava o payload **completo e sem parsing** na Delta Table `bronze.raw_forecast`
- O parsing do array aninhado é feito no notebook `04` (Silver) — princípio Bronze

**Por que este endpoint é excelente para engenharia de dados:**  
A resposta contém um JSON aninhado com `{"list": [...40 objetos...], "city": {...}}`.  
Isso força você a praticar `explode()` e `from_json()` com StructType no notebook Silver.

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — Imports e secret carregados
- [ ] Célula 2 — Função `fetch_forecast` testada com 1 cidade
- [ ] Célula 3 — Loop sobre 4 cidades executado
- [ ] Célula 4 — Tabela `raw_forecast` criada com 4 linhas
- [ ] Célula 5 — Payload inspecionado: chave `"list"` presente com array aninhado
- [ ] Re-executei e count chegou a 8 (append correto)

---
## Célula 1 — Imports e Configuração

In [0]:
import requests
import json
import time
import uuid
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

dbutils.widgets.text("cities",  "Sao Paulo,Rio de Janeiro,Curitiba,Manaus", "Cidades")
dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")

CITIES       = [c.strip() for c in dbutils.widgets.get("cities").split(",")]
CATALOG      = dbutils.widgets.get("catalog")
BRONZE_TABLE = f"{CATALOG}.bronze.raw_forecast"
VOLUME_PATH  = f"/Volumes/{CATALOG}/bronze/landing"

try:
    API_KEY = dbutils.secrets.get(scope="openweather", key="api_key")
    print("✅ API key carregada via Databricks Secrets.")
except Exception:
    API_KEY = ""
    print("⚠️  Secret não encontrado. Configure o scope 'openweather' conforme o notebook 00.")

print(f"Cidades  : {CITIES}")
print(f"Destino  : {BRONZE_TABLE}")

---
## Célula 2 — Função de Extração do Forecast

**Diferença do endpoint `/forecast` vs `/weather`:**
- `/weather` → 1 objeto JSON com o clima atual
- `/forecast` → 1 objeto com `city` (metadados) + `list` (array de 40 previsões de 3h em 3h)

Na Bronze guardamos **tudo como string** — o `explode()` do array acontece na Silver.

In [0]:
BASE_URL = "https://api.openweathermap.org/data/2.5/forecast"

def fetch_forecast(city: str, api_key: str) -> dict:
    """
    Chama o endpoint /forecast e retorna o payload completo como string JSON.
    Erros são capturados — nunca propagados como exceção.
    """
    safe_url = f"{BASE_URL}?q={city}&units=metric"
    try:
        response = requests.get(
            BASE_URL,
            params={"q": city, "appid": api_key, "units": "metric"},
            timeout=10
        )
        response.raise_for_status()
        return {
            "city"        : city,
            "raw_payload" : json.dumps(response.json(), ensure_ascii=False),
            "_status"     : "success",
            "_error_msg"  : None,
            "_source_url" : safe_url,
        }
    except requests.exceptions.ConnectionError as e:
        return {"city": city, "raw_payload": None, "_status": "error",
                "_error_msg": f"ConnectionError: {str(e)[:200]}", "_source_url": safe_url}
    except Exception as e:
        return {"city": city, "raw_payload": None, "_status": "error",
                "_error_msg": str(e)[:200], "_source_url": safe_url}

# Teste isolado
test = fetch_forecast("Sao Paulo", API_KEY)
print(f"Status : {test['_status']}")
if test['_status'] == 'success':
    payload = json.loads(test['raw_payload'])
    print(f"Cidade  : {payload['city']['name']} ({payload['city']['country']})")
    print(f"Previsões no array 'list': {len(payload['list'])} slots de 3h")
    print(f"Primeira previsão: {payload['list'][0]['dt_txt']} → {payload['list'][0]['main']['temp']} °C")
else:
    print(f"Erro: {test['_error_msg']}")
    print("⚠️  Se ConnectionError: use o Plano B — script local + upload para Volume.")

---
## Célula 3 — Loop e Montagem do Batch

In [0]:
BATCH_ID     = str(uuid.uuid4())
INGESTION_TS = datetime.now(timezone.utc)

records = []
for city in CITIES:
    result = fetch_forecast(city, API_KEY)
    records.append({
        "city"                 : result["city"],
        "raw_payload"          : result["raw_payload"],
        "_status"              : result["_status"],
        "_error_msg"           : result["_error_msg"],
        "_source_url"          : result["_source_url"],
        "_batch_id"            : BATCH_ID,
        "_ingestion_timestamp" : INGESTION_TS,
    })
    icon = "✅" if result["_status"] == "success" else "❌"
    print(f"{icon} {city:<20} → {result['_status']}")
    time.sleep(1)

print(f"\nBatch {BATCH_ID[:8]}... — {sum(1 for r in records if r['_status']=='success')}/{len(CITIES)} sucessos")

---
## Célula 4 — Gravar na Delta Table Bronze

In [0]:
schema = StructType([
    StructField("city",                 StringType(),    True),
    StructField("raw_payload",          StringType(),    True),
    StructField("_status",              StringType(),    False),
    StructField("_error_msg",           StringType(),    True),
    StructField("_source_url",          StringType(),    True),
    StructField("_batch_id",            StringType(),    False),
    StructField("_ingestion_timestamp", TimestampType(), False),
])

df_bronze = spark.createDataFrame(records, schema=schema)

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"✅ Gravado em: {BRONZE_TABLE}")
df_bronze.select("city", "_status", "_ingestion_timestamp").show(truncate=False)

---
## Célula 5 — Inspeção do Payload Aninhado

Confirme que:
1. O campo `raw_payload` está preenchido como string JSON
2. O JSON contém a chave `"list"` com 40 objetos
3. Cada objeto tem `dt`, `main.temp`, `weather[0].main`, etc.

**Isso é o que o notebook `04_process_silver_forecast` vai explodir com `explode()`.**

In [0]:
%sql
SELECT
    city,
    _status,
    _ingestion_timestamp,
    -- Extrai o número de previsões no array list[] — deve ser 40
    SIZE(FROM_JSON(raw_payload, 'STRUCT<list: ARRAY<STRING>>').list) AS num_previsoes,
    -- Prévia do payload (primeiros 400 chars)
    LEFT(raw_payload, 400) AS payload_preview
FROM workshop_weather.bronze.raw_forecast
WHERE _status = 'success'
ORDER BY _ingestion_timestamp DESC
LIMIT 4;

In [0]:
%sql
-- Contagem geral da tabela
SELECT
    COUNT(*)                  AS total_registros,
    COUNT(DISTINCT _batch_id) AS total_batches,
    SUM(CASE WHEN _status='success' THEN 1 ELSE 0 END) AS sucesso,
    SUM(CASE WHEN _status='error'   THEN 1 ELSE 0 END) AS erros
FROM workshop_weather.bronze.raw_forecast;

---
## Plano B — Forecast com arquivos locais

Se a API estiver bloqueada, use este script localmente para gerar os arquivos de forecast:

```python
# fetch_forecast_local.py — execute no seu computador
import requests, json, os, time

API_KEY = "SUA_API_KEY_AQUI"
CITIES  = ["Sao Paulo", "Rio de Janeiro", "Curitiba", "Manaus"]
os.makedirs("weather_data/forecast", exist_ok=True)

for city in CITIES:
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/forecast",
        params={"q": city, "appid": API_KEY, "units": "metric"},
        timeout=10
    )
    fname = f"weather_data/forecast/{city.replace(' ','_').lower()}.json"
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(r.json(), f, ensure_ascii=False)
    print(f"✓ {city} → {fname}  ({len(r.json()['list'])} previsões)")
    time.sleep(1)
```

Depois faça upload para `/Volumes/workshop_weather/bronze/landing/sample/forecast/` e use `COPY INTO` ou Auto Loader conforme demonstrado no notebook `01`.

---
## ✅ Checklist Final — Módulo 1B concluído?

- [ ] `raw_forecast` existe em `workshop_weather.bronze`
- [ ] `COUNT(*)` retorna múltiplo de 4
- [ ] `num_previsoes` mostra ~40 para cada cidade com sucesso
- [ ] `payload_preview` contém o JSON com chaves `city` e `list`
- [ ] Você entende por que **não parseamos** o JSON aqui — isso é responsabilidade da Silver

**Próximo: `03_process_silver_current_weather.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*